In [ ]:
import sys
import cv2, os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
sys.path.append("d:/USDA-NIFA/Code")
os.chdir("d:/USDA-NIFA/Code")
os.getcwd()

'e:\\Desktop\\USDA-NIFA\\Code'

In [9]:
aoi_id = 1
aoi_path = "091425_Wallpe/Processed/AOI"
gdf = gpd.read_file(os.path.join(aoi_path, f"AOI_{aoi_id}.geojson"))

In [11]:
def warpImages(img1, img2, H):
    rows1, cols1 = img1.shape[:2]
    rows2, cols2 = img2.shape[:2]

    list_of_points_1 = np.float32([[0,0], [0, rows1],[cols1, rows1], [cols1, 0]]).reshape(-1, 1, 2) #coordinates of a reference image
    temp_points = np.float32([[0,0], [0,rows2], [cols2,rows2], [cols2,0]]).reshape(-1,1,2) #coordinates of second image

    # When we have established a homography we need to warp perspective
    # Change field of view
    list_of_points_2 = cv2.perspectiveTransform(temp_points, H)#calculate the transformation matrix

    list_of_points = np.concatenate((list_of_points_1,list_of_points_2), axis=0)

    [x_min, y_min] = np.int32(list_of_points.min(axis=0).ravel() - 0.5)
    [x_max, y_max] = np.int32(list_of_points.max(axis=0).ravel() + 0.5)
    
    translation_dist = [-x_min,-y_min]
    
    H_translation = np.array([[1, 0, translation_dist[0]], [0, 1, translation_dist[1]], [0, 0, 1]])

    output_img = cv2.warpPerspective(img2, H_translation.dot(H), (x_max-x_min, y_max-y_min))
    output_img[translation_dist[1]:rows1+translation_dist[1], translation_dist[0]:cols1+translation_dist[0]] = img1

    return output_img

In [17]:
img_list = []
tiff_path = "091425_Wallpe/Processed/capture"
for _, row in gdf.iterrows():
    img_list.append(cv2.imread(os.path.join(tiff_path, row['image_name'] + ".tif")))

In [16]:
sift = cv2.SIFT_create()
while True:
    img1 = img_list.pop(0)
    img2 = img_list.pop(0)
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    matcher = cv2.BFMatcher_create(cv2.NORM_HAMMING)
    matches = matcher.match(des1, des2)
    matches = sorted(matches, key = lambda x:x.distance)
    img3 = cv2.drawMatches(img1, kp1, img2, kp2, matches[:10], None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.imshow(img3)
    plt.show()
    good = []
    for m, n in matches:
        if m.distance < 0.75*n.distance:
            good.append(m)

    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)  
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
    result = warpImages(img1, img2, H)
    img_list.insert(0, result)

    if len(img_list) == 1:
        break

error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'detectAndCompute'
> Overload resolution failed:
>  - image is not a numpy array, neither a scalar
>  - Expected Ptr<cv::UMat> for argument 'image'
